In [13]:
# Jalankan cell ini pertama kali untuk memuat semua library yang dibutuhkan
%pip install -r requirements.txt

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses, metrics
import numpy as np
import os

print(f"TensorFlow Version: {tf.__version__}")

Note: you may need to restart the kernel to use updated packages.
TensorFlow Version: 2.21.0



[notice] A new release of pip available: 22.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
# Mendefinisikan 20 kelas secara spesifik sesuai dokumen Project Plan Heartz
# (5 Vokal Dasar + 15 Konsonan Bilabial)
CLASSES = [
    "A", "I", "U", "E", "O",
    "Ba", "Bi", "Bu", "Be", "Bo",
    "Pa", "Pi", "Pu", "Pe", "Po",
    "Ma", "Mi", "Mu", "Me", "Mo"
]

NUM_CLASSES = len(CLASSES)
print(f"Total Kelas Target: {NUM_CLASSES}")

Total Kelas Target: 20


In [15]:
# ======================================================================
# Load dataset .wav dari folder clean (hasil tim Data Science).
# Struktur yang dipakai: data-science/dataset/clean/<KELAS>/*.wav
# Notebook ini akan mengambil 20 kelas yang ada di variabel CLASSES.
# Folder lain (mis. Be2/Be3/O2/O3, dll) akan di-skip.
# ======================================================================

from pathlib import Path
import random

BATCH_SIZE = 32
SAMPLE_RATE = 16000  # Target 1 detik audio
SEED = 42
VAL_SPLIT = 0.2

def _find_project_root(start_dir: Path) -> Path:
    current = start_dir.resolve()
    while True:
        if (current / "data-science").exists():
            return current
        if current == current.parent:
            break
        current = current.parent
    raise FileNotFoundError("Tidak menemukan folder 'data-science' saat mencari project root.")

PROJECT_ROOT = _find_project_root(Path.cwd())
DATASET_DIR = PROJECT_ROOT / "data-science" / "dataset" / "clean"
if not DATASET_DIR.exists():
    raise FileNotFoundError(f"Dataset folder tidak ditemukan: {DATASET_DIR}")

available_class_dirs = sorted([p.name for p in DATASET_DIR.iterdir() if p.is_dir()])
missing = [c for c in CLASSES if c not in available_class_dirs]
if missing:
    raise FileNotFoundError(f"Folder kelas berikut tidak ditemukan di {DATASET_DIR}: {missing}")

ignored = sorted(set(available_class_dirs) - set(CLASSES))
if ignored:
    print("Info: folder kelas yang di-skip (tidak ada di CLASSES):", ignored)

# Stratified split per kelas agar train/val seimbang
rng = random.Random(SEED)
train_paths, train_labels = [], []
val_paths, val_labels = [], []

for label_idx, class_name in enumerate(CLASSES):
    wavs = sorted((DATASET_DIR / class_name).glob("*.wav"))
    if not wavs:
        raise FileNotFoundError(f"Tidak ada file .wav di folder: {DATASET_DIR / class_name}")
    wavs = [str(p) for p in wavs]
    rng.shuffle(wavs)
    split_idx = int(len(wavs) * (1 - VAL_SPLIT))
    cls_train = wavs[:split_idx]
    cls_val = wavs[split_idx:]
    train_paths.extend(cls_train)
    train_labels.extend([label_idx] * len(cls_train))
    val_paths.extend(cls_val)
    val_labels.extend([label_idx] * len(cls_val))

print(f"Dataset siap: train={len(train_paths)} file | val={len(val_paths)} file")

AUTOTUNE = tf.data.AUTOTUNE

def _load_and_prepare_wav(path, label):
    audio_bytes = tf.io.read_file(path)
    audio, sample_rate = tf.audio.decode_wav(audio_bytes, desired_channels=1)
    audio = tf.squeeze(audio, axis=-1)  # [n]
    # Dataset clean diasumsikan sudah 16kHz; fail-fast kalau tidak sesuai.
    tf.debugging.assert_equal(sample_rate, SAMPLE_RATE, message="Sample rate harus 16000 Hz")
    # Pastikan panjang fix = SAMPLE_RATE (truncate/pad)
    audio = audio[:SAMPLE_RATE]
    pad_len = SAMPLE_RATE - tf.shape(audio)[0]
    audio = tf.cond(pad_len > 0, lambda: tf.pad(audio, [[0, pad_len]]), lambda: audio)
    audio.set_shape([SAMPLE_RATE])
    label_oh = tf.one_hot(label, depth=NUM_CLASSES)
    return audio, label_oh

train_dataset = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_dataset = train_dataset.shuffle(buffer_size=min(len(train_paths), 2000), seed=SEED, reshuffle_each_iteration=True)
train_dataset = train_dataset.map(_load_and_prepare_wav, num_parallel_calls=AUTOTUNE)
train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
val_dataset = val_dataset.map(_load_and_prepare_wav, num_parallel_calls=AUTOTUNE)
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)

# Quick sanity check
for x_batch, y_batch in train_dataset.take(1):
    print("Contoh batch:", x_batch.shape, y_batch.shape)

Info: folder kelas yang di-skip (tidak ada di CLASSES): ['Be2', 'Be3', 'Bo2', 'Bo3', 'Me2', 'Me3', 'O2', 'O3']
Dataset siap: train=4870 file | val=1226 file
Contoh batch: (32, 16000) (32, 20)


In [16]:
# Arsitektur Layer khusus untuk mengubah gelombang suara 1D menjadi matriks Spectrogram 2D
# Ini membuktikan kita tidak bergantung pada library eksternal saat tahap inference
@tf.keras.utils.register_keras_serializable()
class MelSpectrogramLayer(layers.Layer):
    def __init__(self, frame_length=255, frame_step=128, **kwargs):
        super(MelSpectrogramLayer, self).__init__(**kwargs)
        self.frame_length = frame_length
        self.frame_step = frame_step

    def call(self, audio_1d):
        # Proses Short-Time Fourier Transform (STFT)
        stft = tf.signal.stft(audio_1d, 
                              frame_length=self.frame_length, 
                              frame_step=self.frame_step)
        spectrogram = tf.abs(stft)
        
        # Tambah dimensi channel (batch, height, width, 1) agar diterima CNN 2D
        spectrogram = tf.expand_dims(spectrogram, axis=-1)
        return spectrogram
        
    def get_config(self):
        config = super(MelSpectrogramLayer, self).get_config()
        config.update({
            "frame_length": self.frame_length,
            "frame_step": self.frame_step,
        })
        return config

print("Custom Layer Spectrogram Siap!")

Custom Layer Spectrogram Siap!


In [17]:
# Membangun arsitektur CNN 2D dari awal tanpa Pre-Trained Model
def build_functional_cnn():
    inputs = tf.keras.Input(shape=(SAMPLE_RATE,), name="audio_raw_1d")
    
    # Konversi on-the-fly
    x = MelSpectrogramLayer(name="spectrogram_conversion")(inputs)
    
    # Layer Konvolusi (Feature Extraction)
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    # Fully Connected Layer
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5, name="dropout_mitigasi_overfitting")(x)
    
    # Output Layer (20 Kelas Suku Kata)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', name="klasifikasi_akhir")(x)
    
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="Heartz_CNN_Functional")
    return model

model = build_functional_cnn()
model.summary()

Model: "Heartz_CNN_Functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ audio_raw_1d (InputLayer)       │ (None, 16000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spectrogram_conversion          │ (None, 124, 129, 1)    │             0 │
│ (MelSpectrogramLayer)           │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 124, 129, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 62, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 62, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 31, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 31, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 15, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 30720)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │     3,932,288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_mitigasi_overfitting    │ (None, 128)            │             0 │
│ (Dropout)                       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ klasifikasi_akhir (Dense)       │ (None, 20)             │         2,580 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,027,540 (15.36 MB)

 Trainable params: 4,027,540 (15.36 MB)

 Non-trainable params: 0 (0.00 B)

In [18]:
# Custom Loss Function: Label Smoothing diterapkan agar penalti lebih stabil
loss_function = losses.CategoricalCrossentropy(label_smoothing=0.1)
optimizer = optimizers.Adam(learning_rate=0.001)

# Inisiasi Metrik Evaluasi
train_acc_metric = metrics.CategoricalAccuracy()
val_acc_metric = metrics.CategoricalAccuracy()

print("Fungsi Loss dan Optimizer siap!")

Fungsi Loss dan Optimizer siap!


In [19]:
# Custom Step untuk Training
@tf.function
def train_step(x_batch, y_batch):
    with tf.GradientTape() as tape:
        logits = model(x_batch, training=True)
        loss_value = loss_function(y_batch, logits)
    
    # Backpropagation manual
    grads = tape.gradient(loss_value, model.trainable_weights)
    optimizer.apply_gradients(zip(grads, model.trainable_weights))
    
    train_acc_metric.update_state(y_batch, logits)
    return loss_value

# Custom Step untuk Validation
@tf.function
def val_step(x_batch, y_batch):
    logits = model(x_batch, training=False)
    loss_value = loss_function(y_batch, logits)
    val_acc_metric.update_state(y_batch, logits)
    return loss_value

# Parameter Early Stopping Manual
EPOCHS = 20
patience = 3
wait = 0
best_val_acc = 0.0
best_weights = model.get_weights()  # fallback kalau val_acc tidak pernah membaik

print("Memulai Custom Training Loop...")

for epoch in range(EPOCHS):
    print(f"\n--- Epoch {epoch+1}/{EPOCHS} ---")
    
    # 1. Fase Training
    for step, (x_batch_train, y_batch_train) in enumerate(train_dataset):
        loss_val = train_step(x_batch_train, y_batch_train)
        
    train_acc = train_acc_metric.result()
    
    # 2. Fase Validation
    for x_batch_val, y_batch_val in val_dataset:
        val_loss = val_step(x_batch_val, y_batch_val)
        
    val_acc = val_acc_metric.result()
    
    print(f"Train Accuracy: {train_acc:.4f} | Val Accuracy: {val_acc:.4f}")
    
    # Reset metrik untuk epoch berikutnya
    train_acc_metric.reset_state()
    val_acc_metric.reset_state()
    
    # 3. Logika Early Stopping
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        wait = 0
        # Simpan bobot terbaik ke memori
        best_weights = model.get_weights()
        print(" -> Akurasi meningkat! Bobot model diperbarui.")
    else:
        wait += 1
        if wait >= patience:
            print(f" -> Early Stopping diaktifkan! Val Accuracy tidak membaik selama {patience} epoch.")
            model.set_weights(best_weights)  # Kembalikan ke bobot terbaik
            break

print("Training Selesai!")

Memulai Custom Training Loop...

--- Epoch 1/20 ---
Train Accuracy: 0.8010 | Val Accuracy: 0.9812
 -> Akurasi meningkat! Bobot model diperbarui.

--- Epoch 2/20 ---
Train Accuracy: 0.9688 | Val Accuracy: 0.9861
 -> Akurasi meningkat! Bobot model diperbarui.

--- Epoch 3/20 ---
Train Accuracy: 0.9893 | Val Accuracy: 0.9951
 -> Akurasi meningkat! Bobot model diperbarui.

--- Epoch 4/20 ---
Train Accuracy: 0.9926 | Val Accuracy: 0.9959
 -> Akurasi meningkat! Bobot model diperbarui.

--- Epoch 5/20 ---
Train Accuracy: 0.9959 | Val Accuracy: 1.0000
 -> Akurasi meningkat! Bobot model diperbarui.

--- Epoch 6/20 ---
Train Accuracy: 0.9977 | Val Accuracy: 1.0000

--- Epoch 7/20 ---
Train Accuracy: 0.9984 | Val Accuracy: 1.0000

--- Epoch 8/20 ---
Train Accuracy: 0.9979 | Val Accuracy: 1.0000
 -> Early Stopping diaktifkan! Val Accuracy tidak membaik selama 3 epoch.
Training Selesai!


In [20]:
# Membuat folder jika belum ada
os.makedirs("saved_models", exist_ok=True)

# Simpan model final dalam format H5 agar bisa ditarik oleh FastAPI nanti
MODEL_PATH = "saved_models/heartz_model_final.h5"
model.save(MODEL_PATH)

print(f"Model berhasil diekspor dan siap di-deploy ke: {MODEL_PATH}")

Model berhasil diekspor dan siap di-deploy ke: saved_models/heartz_model_final.h5
